In [1]:
import matplotlib.pyplot as plt
import sunpy.map
import os
import re
from datetime import datetime
from matplotlib.patches import Wedge
from astropy.visualization import ImageNormalize, LinearStretch
from astropy.coordinates import SkyCoord
import astropy.units as u

%matplotlib notebook

In [3]:
file_path = "E:/python/projects/alfven/data/hrieuvopn/"
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]
def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

files = sorted(files, key=extract_datetime)

In [ ]:
m_seq = sunpy.map.Map(files, sequence=True)

fig  = plt.figure()
ax = fig.add_subplot(projection=m_seq.maps[0])
ani = m_seq.plot(axes=ax, norm=ImageNormalize(stretch=LinearStretch(), vmin=0, vmax=2500), interval=50)
plt.colorbar()

In [ ]:
def cut_map(hri, bottom_left, top_right):
    if type(hri) is str:
        hri = sunpy.map.Map(hri)
    bottom_left = SkyCoord(bottom_left[0]*u.arcsec, bottom_left[1]*u.arcsec, frame=hri.coordinate_frame)
    top_right = SkyCoord(top_right[0]*u.arcsec, top_right[1]*u.arcsec, frame=hri.coordinate_frame)
    
    submap = hri.submap(bottom_left, top_right=top_right)
    return submap

submaps = []
top_right = (100, -2900)
bottom_left = (-100, -3100)
for euimap in m_seq.maps:
    submaps.append(cut_map(euimap, bottom_left, top_right))
    
submap_seq = sunpy.map.Map(submaps, sequence=True)

In [ ]:
fig  = plt.figure()
ax = fig.add_subplot(projection=submap_seq.maps[0])
ani = submap_seq.plot(axes=ax, norm=ImageNormalize(stretch=LinearStretch(), vmin=0, vmax=2500), interval=50, cmap='sdoaia171')
plt.colorbar()

In [ ]:
ani.save('./fig/hri_ani.gif')